In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path(r"D:\BAPETEN\BAPETEN PROJEK\neta-eksplorasi")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from services.google_services_own_account import (
    connect_google,
    read_excel,
)

drive, sheets = connect_google()

In [ ]:
import pandas as pd

ID_FILE_EXCEL = "1vqxcE_9L_T4VZ3z7g-BC8fp7StIMVSX_"

df_excel = read_excel(
    ID_FILE_EXCEL,
    drive
)

df = pd.read_excel(
        df_excel,
        sheet_name="raw_data",
        engine="openpyxl"
    )

df.head()



Berhasil


,No,Nama Instansi /Fasilitas,Alamat,Lokasi Pemanfaatan,Nomor LHI,Kelompok Kegiatan,Tanggal Inspeksi,Kategori Temuan,Status Bundling/ Temuan,Kode Temuan,Temuan dan Rekomendasi,Uraian Temuan,Uraian Rekomendasi,Tgl Batas Tindak Lanjut,Uraian Tindak Lanjut,Tgl Tindak Lanjut
0,1,Bhayangkara Tk.III Manado,Jl. Sam Ratulangi No. 326,"Jl. Sam Ratulangi No. 326,Manado,Sulawesi Utara",00053.01.12082.01.1.0426,Radiologi Diagnostik & Intervensional,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,Rumah Sakit Bhayangkara Tk. III Manado memilik...,-Rumah Sakit Bhayangkara Tk.III Manado dilaran...,2026-08-07 00:00:00,NaN,2026-05-05
1,2,BLU RSUP. Prof. Dr. R.D. Kandou Manado,Jl. Raya Tanawangko No 56 Manado Sulawesi Utara,Jl. Raya Tanawangko No 56 Manado Sulawesi Utar...,00073.01.00475.01.1.0426,Radiologi Diagnostik & Intervensional,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,- Terdapat 13 (tiga belas) Alat Radiologi Diag...,-BLU RSUP. Prof. Dr. R.D. Kandou Manado dilara...,2026-08-28 00:00:00,Surat Penonaktifan alat,2026-05-05
2,3,Conch North Sulawesi Cement,"The Suites Tower Lt. 19, Jl. Boulevard Pantai ...","The Suites Tower Lt. 19, Jl. Boulevard Pantai ...",00069.01.10409.08.1.0426,Gauging,2026-04-20 s/d 2026-04-24,1,tertutup,1.01.01,Tidak memiliki izin,"Terdapat 2 (dua) SRP yang belum memiliki izin,...",-PT. Conch North Sulawesi Cement dilarang meng...,0000-00-00,Untuk SRP Cf 252 dengan Kode Sumber 0191052 su...,2026-06-15
3,4,Vale Indonesia Tbk.,"The Energy Building, 31st Floor, SCBD Lot 11 A...","The Energy Building, 31st Floor, SCBD Lot 11 A...",00077.01.07945.20.1.0426,Fluoroskopi Bagasi,2026-04-20 s/d 2026-04-24,1,tertutup,1.01.01,Tidak memiliki izin,"Berdasarkan pemaparan pihak PT Vale Indonesia,...",-Pelaku usaha agar segera mengajukan permohona...,0000-00-00,Menindaklanjuti Surat Hasil Inspeksi Nomor : T...,2026-09-15
4,5,Vale Indonesia Tbk.,"The Energy Building, 31st Floor, SCBD Lot 11 A...","The Energy Building, 31st Floor, SCBD Lot 11 A...",00076.01.07945.08.1.0426,Gauging,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,"Berdasarkan pemaparan pihak PT Vale Indonesia,...",-Pelaku usaha agar segera mengajukan permohona...,2026-09-25 00:00:00,1. Mengajukan permohonan izin pemanfaatan selu...,2026-08-21


KeyError: 'Tahun'

In [12]:
print(df["Tgl Batas Tindak Lanjut"].dtype)

object


In [3]:
df["Tgl Batas Tindak Lanjut"] = pd.to_datetime(
    df["Tgl Batas Tindak Lanjut"],
    errors="coerce"
).dt.date

In [13]:
print(df["Tgl Batas Tindak Lanjut"].dtype)

object


In [15]:
import re

provinsi_indonesia = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau",
    "Kepulauan Riau", "Jambi", "Sumatera Selatan", "Kepulauan Bangka Belitung",
    "Bengkulu", "Lampung", "DKI Jakarta", "Banten", "Jawa Barat",
    "Jawa Tengah", "DI Yogyakarta", "Jawa Timur", "Bali",
    "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat",
    "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur",
    "Kalimantan Utara", "Sulawesi Utara", "Gorontalo",
    "Sulawesi Tengah", "Sulawesi Barat", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Maluku", "Maluku Utara", "Papua",
    "Papua Barat", "Papua Selatan", "Papua Tengah",
    "Papua Pegunungan", "Papua Barat Daya"
]

# Alias untuk variasi penulisan alamat
alias_provinsi = {
    "D.I. Yogyakarta": "DI Yogyakarta",
    "Daerah Istimewa Yogyakarta": "DI Yogyakarta",
    "D.I. Yogyakarta": "DI Yogyakarta",
    "DKI Jakarta": "DKI Jakarta",
    "Daerah Khusus Jakarta": "DKI Jakarta",
}

# Cocokkan nama terpanjang terlebih dahulu
pola = re.compile(
    r"(?<!\w)(" +
    "|".join(
        re.escape(p)
        for p in sorted(
            provinsi_indonesia + list(alias_provinsi.keys()),
            key=len,
            reverse=True
        )
    ) +
    r")(?!\w)",
    flags=re.IGNORECASE
)

def cari_provinsi(alamat):
    if not isinstance(alamat, str):
        return None

    hasil = pola.search(alamat)
    if hasil is None:
        return None

    nama = hasil.group(0)

    for alias, nama_resmi in alias_provinsi.items():
        if nama.lower() == alias.lower():
            return nama_resmi

    # Samakan dengan ejaan pada daftar
    return next(
        (p for p in provinsi_indonesia if p.lower() == nama.lower()),
        None
    )

df["Provinsi"] = df["Lokasi Pemanfaatan"].apply(cari_provinsi)

df = df.dropna(subset=["Provinsi"]).reset_index(drop=True)


In [12]:
df.head()

,No,Nama Instansi /Fasilitas,Alamat,Lokasi Pemanfaatan,Nomor LHI,Kelompok Kegiatan,Tanggal Inspeksi,Kategori Temuan,Status Bundling/ Temuan,Kode Temuan,Temuan dan Rekomendasi,Uraian Temuan,Uraian Rekomendasi,Tgl Batas Tindak Lanjut,Uraian Tindak Lanjut,Tgl Tindak Lanjut,Provinsi
0,1,Bhayangkara Tk.III Manado,Jl. Sam Ratulangi No. 326,"Jl. Sam Ratulangi No. 326,Manado,Sulawesi Utara",00053.01.12082.01.1.0426,Radiologi Diagnostik & Intervensional,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,Rumah Sakit Bhayangkara Tk. III Manado memilik...,-Rumah Sakit Bhayangkara Tk.III Manado dilaran...,2026-08-07 00:00:00,NaN,2026-05-05,Sulawesi Utara
1,2,BLU RSUP. Prof. Dr. R.D. Kandou Manado,Jl. Raya Tanawangko No 56 Manado Sulawesi Utara,Jl. Raya Tanawangko No 56 Manado Sulawesi Utar...,00073.01.00475.01.1.0426,Radiologi Diagnostik & Intervensional,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,- Terdapat 13 (tiga belas) Alat Radiologi Diag...,-BLU RSUP. Prof. Dr. R.D. Kandou Manado dilara...,2026-08-28 00:00:00,Surat Penonaktifan alat,2026-05-05,Sulawesi Utara
2,3,Conch North Sulawesi Cement,"The Suites Tower Lt. 19, Jl. Boulevard Pantai ...","The Suites Tower Lt. 19, Jl. Boulevard Pantai ...",00069.01.10409.08.1.0426,Gauging,2026-04-20 s/d 2026-04-24,1,tertutup,1.01.01,Tidak memiliki izin,"Terdapat 2 (dua) SRP yang belum memiliki izin,...",-PT. Conch North Sulawesi Cement dilarang meng...,0000-00-00,Untuk SRP Cf 252 dengan Kode Sumber 0191052 su...,2026-06-15,DKI Jakarta
3,4,Vale Indonesia Tbk.,"The Energy Building, 31st Floor, SCBD Lot 11 A...","The Energy Building, 31st Floor, SCBD Lot 11 A...",00077.01.07945.20.1.0426,Fluoroskopi Bagasi,2026-04-20 s/d 2026-04-24,1,tertutup,1.01.01,Tidak memiliki izin,"Berdasarkan pemaparan pihak PT Vale Indonesia,...",-Pelaku usaha agar segera mengajukan permohona...,0000-00-00,Menindaklanjuti Surat Hasil Inspeksi Nomor : T...,2026-09-15,DKI Jakarta
4,5,Vale Indonesia Tbk.,"The Energy Building, 31st Floor, SCBD Lot 11 A...","The Energy Building, 31st Floor, SCBD Lot 11 A...",00076.01.07945.08.1.0426,Gauging,2026-04-20 s/d 2026-04-24,1,terbuka,1.01.01,Tidak memiliki izin,"Berdasarkan pemaparan pihak PT Vale Indonesia,...",-Pelaku usaha agar segera mengajukan permohona...,2026-09-25 00:00:00,1. Mengajukan permohonan izin pemanfaatan selu...,2026-08-21,DKI Jakarta


In [16]:
import numpy as np
import pandas as pd

def tahun_temuan(file_olah, kategori, tahun="Semua", provinsi="Semua"):

    tanggal = pd.to_datetime(
        file_olah["Tgl Tindak Lanjut"],
        errors="coerce"
    )


    kondisi = file_olah['Kategori Temuan'] == kategori

    if isinstance(tahun, (list, tuple, np.ndarray)):
        if len(tahun) > 0:
            kondisi &= tanggal.dt.year.isin([int(t) for t in tahun])
    elif tahun != "Semua":
        kondisi &= tanggal.dt.year == int(tahun)


    if isinstance(provinsi, (list, tuple, np.ndarray)):
        if len(provinsi) > 0:
            kondisi &= file_olah["Provinsi"].isin(provinsi)
    elif provinsi != "Semua":
        kondisi &= file_olah["Provinsi"] == provinsi

    return len(file_olah.loc[kondisi])


tahun_temuan(df, "1", "2026", "Banten")

0